# Chapter 31 – Worked examples

This notebook runs the examples of [Chapter 31](../index.md) live. The code lives in the modules next to it
(`n1_screen.py`, `grid_data.py`, `n1_batch.py`, `timing_sweep.py`, `compile_count.py`, `control_flow.py`,
`lodf_precision.py`); the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository. Everything runs on CPU here; the same cells run unchanged on a GPU.

In [ ]:
import compile_count
import control_flow
import grid_data
import jax
import jax.numpy as jnp
import lodf_precision
import n1_batch
import n1_screen
import numpy as np
import timing_sweep

print('JAX', jax.__version__, 'on', jax.default_backend(), '- x64 enabled:', jax.config.jax_enable_x64)

## N-1 screening with `jit` and `vmap`

`n1_flows` handles **one** outage and returns a flag instead of raising on a bridge; `screen` is
`jit(vmap(n1_flows))`. On the four-bus grid every row of the result is one outage.

In [ ]:
from_bus, to_bus = jnp.array([0, 0, 1, 1, 2]), jnp.array([1, 2, 2, 3, 3])
x = jnp.array([0.1, 0.2, 0.1, 0.2, 0.1])
p = jnp.array([100.0, 0.0, 0.0, -100.0])
ptdf = n1_screen.ptdf_matrix(from_bus, to_bus, x, 4)
flows, ok = n1_screen.screen(ptdf, from_bus, to_bus, p, jnp.arange(5))
print(jnp.round(flows, 1))
print(ok)

## The same screening on IEEE 14, checked against NumPy

`grid_data.branch_data` pulls the arrays out of pandapower; `numpy_screen` is the Chapter 17 loop. The two
agree to $10^{-9}$ on every outage that does not island the grid (IEEE 14 has one bridge).

In [ ]:
fb, tb, xb, pb = grid_data.branch_data('case14')
ptdf14 = n1_screen.ptdf_matrix(jnp.array(fb), jnp.array(tb), jnp.array(xb), len(pb))
flows14, ok14 = n1_screen.screen(ptdf14, jnp.array(fb), jnp.array(tb), jnp.array(pb), jnp.arange(len(xb)))
ref, ok_ref = grid_data.numpy_screen(np.asarray(ptdf14), fb, tb, pb)
print(
    'bridges:',
    int((~ok14).sum()),
    ' max |JAX - NumPy|:',
    float(np.abs(np.asarray(flows14)[ok_ref] - ref[ok_ref]).max()),
)

## A second batch axis

`vmap` once more, over 100 injection vectors drawn with split PRNG keys: one call, one compiled program, an
array of shape `(100, n_branch, n_branch)` on IEEE 118.

In [ ]:
f118, t118, x118, p118 = (jnp.array(v) for v in grid_data.branch_data('case118'))
ptdf118 = n1_screen.ptdf_matrix(f118, t118, x118, len(p118))
injections = n1_batch.random_injections(jax.random.key(0), p118, 100)
batch, ok_batch = n1_batch.screen_injections(ptdf118, f118, t118, injections, jnp.arange(len(x118)))
print(batch.shape, ok_batch.shape, ' bridges per injection:', int((~ok_batch[0]).sum()))

## Compile time against run time

The first call includes compilation; the median of the next ten is the steady state. Every call waits with
`block_until_ready`. `python timing_sweep.py` runs the four grids of the lab; here only IEEE 118.

In [ ]:
for row in timing_sweep.sweep(('case118',), repeats=5):
    print(f'{row["dtype"]}: first {row["first_s"]:.3f} s, steady {row["steady_s"] * 1e3:.2f} ms')

## Counting compilations

A Python counter inside a jitted function only runs while tracing. A static tuple compiles once per distinct
tuple, a fresh `lambda` compiles every call, and the fixed versions compile once; `jax.no_tracing()` would
raise if they did not.

In [ ]:
sets = compile_count.ten_outage_sets(5, 2)
for outages in sets:
    compile_count.screen_static(ptdf, from_bus, to_bus, p, outages)
print('static tuple:', len(sets), 'distinct tuples ->', compile_count.static_traces[0], 'traces')
for _ in range(3):
    compile_count.screen_then(ptdf, from_bus, to_bus, p, jnp.arange(5), post=lambda f: jnp.abs(f).max(axis=1))  # noqa: B023
print('fresh lambda: 3 calls ->', compile_count.then_traces[0], 'traces')
for _ in range(3):
    compile_count.screen_then(ptdf, from_bus, to_bus, p, jnp.arange(5), post=compile_count.worst_flow)
print('module-level function: 3 more calls ->', compile_count.then_traces[0], 'traces in total')
print('Exercise 1, traces after each call:', compile_count.exercise_1_traces())

## Control flow and sentinels

A Python `if` on a traced value fails under `jit`; `jnp.where` and `lax.cond` both work, but `where`
evaluates both branches (watch the gradient). A padding sentinel equal to the array length is dropped by
`mode="fill"`; `-1` silently reads the last element.

In [ ]:
xs = jnp.array([-1.0, 2.0, -3.0, 5.0])
print('under jit:', control_flow.error_under_jit(xs))
print('where:', jax.jit(control_flow.clip_negative_where)(xs), ' cond:', jax.jit(control_flow.clip_negative_cond)(xs))
print(
    'grad of where-sqrt at -1:',
    jax.grad(control_flow.safe_sqrt_where)(-1.0),
    ' with cond:',
    jax.grad(control_flow.safe_sqrt_cond)(-1.0),
)
values = jnp.array([10.0, 20.0, 30.0, 40.0])
groups = [[0, 1, 2], [3], [1, 3]]
print('sentinel 4 :', control_flow.gather_fill(values, control_flow.pad_indices(groups, 4)).sum(axis=1))
print('sentinel -1:', control_flow.gather_fill(values, control_flow.pad_indices(groups, -1)).sum(axis=1))

## float32 against float64 in the LODF

The float32 error of an LODF column grows as its denominator $|1 - (\mathrm{PTDF}_{b,f_b} - \mathrm{PTDF}_{b,t_b})|$
shrinks: near-bridges amplify round-off.

In [ ]:
study = lodf_precision.precision_study('case118')
ok_s, denom, err = study['ok'], study['abs_denom'], study['error']
print(
    'bridges:',
    int((~ok_s).sum()),
    ' median error |denom| < 0.2:',
    f'{np.median(err[ok_s][denom[ok_s] < 0.2]):.1e}',
    ' |denom| >= 0.5:',
    f'{np.median(err[ok_s][denom[ok_s] >= 0.5]):.1e}',
)